# AETHER Qwen Brain — Kaggle Deployment

Serves one Qwen model behind three endpoints (`/chat`, `/generate`, `/director`)
and exposes it to AETHER over an ngrok tunnel.

**Before you run anything:** set the accelerator to **GPU T4 x2**
(Settings → Accelerator), and add `NGROK_AUTHTOKEN` and `DIRECTOR_API_KEY`
under Add-ons → Secrets.

### A note on the model

Kaggle's GPUs are Turing (T4, compute capability 7.5). That rules out two
things people reach for first:

* **bfloat16** does not exist on Turing, so the server runs in `float16`.
* **compressed-tensors / `pack-quantized` 4-bit** checkpoints — including
  `cyankiwi/Qwen3.8-27B-AWQ-INT4`, despite the name — decode through Marlin
  kernels that require compute capability 8.0. They cannot load on a T4 at any
  setting. That model is also a hybrid linear-attention VLM, which needs newer
  kernels still.

So this notebook runs a genuine **AWQ** checkpoint, which vLLM has a Turing
code path for. Pick your size in the config cell below.

## Stage 1 — Hardware

In [ ]:
import subprocess, sys, torch

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

GPU_COUNT = torch.cuda.device_count()
if GPU_COUNT == 0:
    raise RuntimeError('No GPU. Settings -> Accelerator -> GPU T4 x2.')

caps = []
for i in range(GPU_COUNT):
    p = torch.cuda.get_device_properties(i)
    caps.append(p.major * 10 + p.minor)
    print(f'  GPU {i}: {p.name}  {p.total_memory/1024**3:.1f} GB  sm_{p.major}{p.minor}')

MIN_CAP = min(caps)
# bfloat16 needs sm_80. Everything below it has to be told to use float16, or
# vLLM aborts on the model's own torch_dtype.
DTYPE = 'bfloat16' if MIN_CAP >= 80 else 'float16'
TOTAL_VRAM = sum(torch.cuda.get_device_properties(i).total_memory for i in range(GPU_COUNT)) / 1024**3

print(f'\nCUDA {torch.version.cuda} | torch {torch.__version__}')
print(f'GPUs {GPU_COUNT} | min sm_{MIN_CAP} | dtype -> {DTYPE} | total VRAM {TOTAL_VRAM:.1f} GB')
if GPU_COUNT < 2:
    print('\n[WARN] Only 1 GPU. A 32B model will not fit — pick a smaller one below.')
print('\nStage 1 PASSED')

## Stage 2 — Dependencies (~5-10 min)

In [ ]:
import subprocess, sys

# Pinned on purpose. Installing bleeding-edge transformers from git alongside
# vLLM breaks vLLM: it pins the transformers it was built against, and pip will
# happily satisfy the git URL by replacing it.
PACKAGES = [
    'vllm==0.27.1',
    'fastapi',
    'uvicorn[standard]',
    'pyngrok',
    'openai',
]

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-q', *PACKAGES])

import importlib.metadata as md
for pkg in ('vllm', 'transformers', 'torch', 'pydantic'):
    try:
        print(f'  {pkg:14} {md.version(pkg)}')
    except Exception:
        print(f'  {pkg:14} (not installed)')
print('\nStage 2 PASSED')

## Stage 3 — Configuration

In [ ]:
import os

# Every option here is a real AWQ checkpoint (quant_method 'awq'), which is the
# only 4-bit format with a Turing kernel in vLLM.
#
#   name                    weights   needs        speed on 2x T4
#   Qwen/Qwen3-32B-AWQ      ~19 GB    2 GPUs       slowest, strongest
#   Qwen/Qwen3-14B-AWQ      ~9 GB     1-2 GPUs     ~2.5x faster  <- good default
#   Qwen/Qwen3-8B-AWQ       ~5.5 GB   1 GPU        fastest, weakest
MODEL_CHOICE = 'Qwen/Qwen3-14B-AWQ'

DIRECTOR_MODEL = MODEL_CHOICE
# 8B fits on one card, so sharding it only adds communication overhead.
# Anything larger uses every GPU there is.
DIRECTOR_TP_SIZE = 1 if '8B' in MODEL_CHOICE else min(GPU_COUNT, 2)
DIRECTOR_MAX_LEN = 8192
GPU_MEM_UTIL     = 0.90
MAX_NUM_SEQS     = 4
VLLM_PORT        = 8000
API_PORT         = 8001

try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    NGROK_AUTHTOKEN  = _s.get_secret('NGROK_AUTHTOKEN')
    DIRECTOR_API_KEY = _s.get_secret('DIRECTOR_API_KEY')
except Exception:
    NGROK_AUTHTOKEN  = os.environ.get('NGROK_AUTHTOKEN', '')
    DIRECTOR_API_KEY = os.environ.get('DIRECTOR_API_KEY', 'test-key-change-me')

os.environ['DIRECTOR_MODEL']   = DIRECTOR_MODEL
os.environ['DIRECTOR_API_KEY'] = DIRECTOR_API_KEY
os.environ['VLLM_PORT']        = str(VLLM_PORT)

print('Model   :', DIRECTOR_MODEL)
print('TP size :', DIRECTOR_TP_SIZE)
print('dtype   :', DTYPE)
print('Max len :', DIRECTOR_MAX_LEN)
print('API key :', (DIRECTOR_API_KEY[:4] + '****') if DIRECTOR_API_KEY else 'NOT SET')
print('Ngrok   :', 'configured' if NGROK_AUTHTOKEN else 'NOT SET (localhost only)')

## Stage 4 — Write the director package

In [ ]:
import os
from pathlib import Path
os.makedirs('director', exist_ok=True)
print('[stage 4] cwd:', os.getcwd())

In [ ]:
# director/__init__.py — generated from the package by build_notebook.py
SRC = '# Initialize director module\n'
Path('director/__init__.py').write_text(SRC, encoding='utf-8')
print('  wrote director/__init__.py', len(SRC), 'bytes')

In [ ]:
# director/schema.py — generated from the package by build_notebook.py
SRC = '"""The storyboard contract.\n\nThis mirrors, field for field, what AETHER\'s `parseVideoPlan()` in\npublic/prompts.js actually reads. That parser is unforgiving in two ways worth\nstating up front, because getting either wrong makes the director look broken\nin a way that produces no error message:\n\n  * a scene whose `index` is not a finite number is DROPPED from the result,\n    so an omitted index silently deletes the beat; and\n  * `visualType` is snapped onto AETHER\'s vocabulary, with anything\n    unrecognised falling back to \'stock_video\' — an invented type is not an\n    error, it is a wrong answer that looks like a right one.\n\nBecause these models are handed to vLLM as a JSON schema for guided decoding,\nthe grammar itself is what keeps the model inside the vocabulary. Widening a\nLiteral here widens what the model may emit, so the lists below must stay in\nstep with VISUAL_TYPES / SHOT_TYPES / CAMERA_MOVES / HOST_OVERLAYS in\npublic/prompts.js.\n"""\n\nfrom __future__ import annotations\n\nfrom typing import List, Optional, Literal\n\nfrom pydantic import BaseModel, Field\n\n# Kept in the same order as public/prompts.js so the two can be diffed by eye.\nVisualType = Literal[\n    "stock_video", "stock_photo", "stock_text", "editorial_text",\n    "t2v", "broll", "presenter",\n    "stickman", "whiteboard", "chart", "map", "timeline", "diagram",\n]\n\nShotType = Literal[\n    "Extreme Wide", "Wide", "Medium", "Close Up",\n    "Extreme Close Up", "Over Shoulder", "POV",\n]\n\nCameraMove = Literal[\n    "Static", "Slow Push In", "Dolly In", "Dolly Out", "Pan Left", "Pan Right",\n    "Crane Up", "Crane Down", "Handheld", "Drone",\n]\n\nHostOverlay = Literal["none", "circle", "rect", "corner", "full"]\nTextStyle = Literal["stat", "quote", "title", "emphasis", "callout"]\nTransition = Literal["cut", "dissolve"]\n\n\nclass StockRequirements(BaseModel):\n    """What to search the stock libraries for. Required for the stock_* types."""\n\n    concept: str = Field(\n        description="One sentence naming what this beat is actually about."\n    )\n    queries: List[str] = Field(\n        description=(\n            "3-5 standalone stock-search phrases, written as a stock "\n            "photographer would search: subject + action + setting. Describe "\n            "what the camera sees, never what the narration says. Good: "\n            "\'scientist looking into microscope\'. Bad: \'the consequences of "\n            "inflation\'."\n        ),\n        min_length=1,\n        max_length=6,\n    )\n    fallbackQueries: List[str] = Field(\n        default_factory=list,\n        description="2-3 broader queries to try if the primary queries find nothing.",\n        max_length=4,\n    )\n    subjectCategory: Optional[Literal["HUMAN", "NATURE", "URBAN", "ABSTRACT", "OBJECT"]] = Field(\n        default=None, description="Coarse subject bucket, used to break ties between clips."\n    )\n    minimumDuration: float = Field(\n        default=0,\n        ge=0,\n        description="Seconds the clip must run at minimum. 0 means half the scene duration.",\n    )\n\n\nclass TextOverlay(BaseModel):\n    """Editorial type burned over the beat. Required for stock_text and editorial_text."""\n\n    text: str = Field(\n        description="The words on screen. Under 12 words — this is a caption, not the narration."\n    )\n    emphasis: str = Field(\n        default="",\n        description="The 1-3 words within `text` to set larger or brighter.",\n    )\n    style: TextStyle = Field(default="emphasis", description="Which type treatment to use.")\n\n\nclass Graphic(BaseModel):\n    """Content for a canvas-drawn beat: stickman, whiteboard, chart, map, timeline, diagram."""\n\n    title: str = Field(default="", description="Heading for the card.")\n    subtitle: str = Field(default="", description="Optional second line.")\n    items: List[str] = Field(\n        default_factory=list,\n        max_length=6,\n        description=(\n            "The actual content to typeset, and the format depends on the type: "\n            "\'Label: Number\' pairs for chart, \'Date: Event\' for timeline, place "\n            "names for map, steps for whiteboard, labelled parts for diagram, and "\n            "\'action:expression\' pairs such as \'explain:confident\' for stickman."\n        ),\n    )\n\n\nclass Scene(BaseModel):\n    """One beat. AETHER calls these scenes and they map 1:1 onto storyboard rows."""\n\n    index: int = Field(\n        ge=0,\n        description=(\n            "Zero-based position of this beat, matching the input cue list. "\n            "AETHER discards any scene without a numeric index, so never omit it."\n        ),\n    )\n    visualType: VisualType = Field(description="Which renderer this beat is sent to.")\n\n    stockRequirements: Optional[StockRequirements] = Field(\n        default=None, description="Required for stock_video, stock_photo and stock_text."\n    )\n    textOverlay: Optional[TextOverlay] = Field(\n        default=None, description="Required for stock_text and editorial_text."\n    )\n    graphic: Optional[Graphic] = Field(\n        default=None,\n        description="Required for stickman, whiteboard, chart, map, timeline and diagram.",\n    )\n\n    hostOverlay: HostOverlay = Field(\n        default="none",\n        description=(\n            "Where the channel host sits over the visual. \'full\' only with "\n            "presenter; \'none\' for footage beats and anything emotional."\n        ),\n    )\n    shotType: ShotType = Field(default="Medium", description="Shot scale.")\n    cameraMovement: CameraMove = Field(default="Static", description="Camera move.")\n    motion: str = Field(\n        default="",\n        description="What physically moves in the shot. A footage beat with no motion is an expensive still.",\n    )\n    emotion: str = Field(default="", description="One or two words for the intended mood.")\n    transition: Transition = Field(\n        default="cut", description="How this beat joins the NEXT one."\n    )\n    note: str = Field(\n        default="",\n        description="Continuity warnings, or why this visual choice was made. Keep it short.",\n    )\n\n\nclass VideoPlan(BaseModel):\n    """The whole director answer. Top-level shape read by parseVideoPlan()."""\n\n    strategy: str = Field(\n        default="",\n        description="A sentence or two on the visual approach taken across the video.",\n    )\n    warnings: List[str] = Field(\n        default_factory=list,\n        description="Continuity problems: contradictions in place, time of day or weather.",\n    )\n    scenes: List[Scene] = Field(min_length=1, description="Every beat, in order.")\n\n\n# Retained so `from .schema import Storyboard` keeps working in older notebooks.\nStoryboard = VideoPlan\n\n\ndef _tighten(node: object) -> object:\n    """Forbid unspecified keys everywhere in the generated JSON schema.\n\n    Guided decoding follows the grammar it is handed. Left open, an object\n    permits arbitrary extra keys, and the model will occasionally invent one\n    instead of filling the field we asked for — which then arrives as a beat\n    with no queries. Closing the objects removes the option.\n    """\n    if isinstance(node, dict):\n        if node.get("type") == "object" and "additionalProperties" not in node:\n            node["additionalProperties"] = False\n        for value in node.values():\n            _tighten(value)\n    elif isinstance(node, list):\n        for value in node:\n            _tighten(value)\n    return node\n\n\ndef get_json_schema() -> dict:\n    """The JSON schema handed to vLLM for structured output."""\n    return _tighten(VideoPlan.model_json_schema())\n'
Path('director/schema.py').write_text(SRC, encoding='utf-8')
print('  wrote director/schema.py', len(SRC), 'bytes')

In [ ]:
# director/prompts.py — generated from the package by build_notebook.py
SRC = '"""System prompts for the AETHER brain.\n\nTwo prompts, because the model does two different jobs. The general one is for\nconversation and open-ended generation; the director one is for producing a\nstoryboard against a grammar. Keeping them apart stops storyboard rules from\nleaking into a chat reply and vice versa.\n"""\n\nAETHER_SYSTEM_PROMPT = (\n    "You are the central intelligence for AETHER, a long-form video production "\n    "studio. You help with scriptwriting, research, structure, SEO and "\n    "storyboarding.\\n"\n    "\\n"\n    "AETHER builds videos from real stock footage (Pixabay, Pexels) and from "\n    "graphics it typesets itself. It does not generate video from a text "\n    "prompt. So when you describe a visual, describe something that could "\n    "actually be found in a stock library or drawn as a chart — never invent "\n    "an asset, a statistic, or an event.\\n"\n    "\\n"\n    "Write plainly. No preamble, no restating the question, no offers of "\n    "further help unless asked."\n)\n\nDIRECTOR_SYSTEM_PROMPT = (\n    "You are the Visual Director for a long-form video. For every beat of the "\n    "script you decide what the audience SEES at that moment, and why that "\n    "choice serves the story.\\n"\n    "\\n"\n    "Your answer is consumed by software, not by a person. It must match the "\n    "supplied JSON schema exactly. No markdown, no commentary, no code fence.\\n"\n    "\\n"\n    "ALWAYS PREFER THE SIMPLEST VISUAL THAT EXPLAINS THE IDEA. A drawn or "\n    "typeset visual renders instantly, looks identical on every run, and stays "\n    "editable. A searched clip depends on what the library happens to hold. "\n    "Choose in this order, and only move down when the option above genuinely "\n    "cannot communicate the point:\\n"\n    "  1. chart / map / timeline  - the beat carries numbers, places or dates\\n"\n    "  2. whiteboard / diagram    - a process, mechanism or labelled structure\\n"\n    "  3. stickman                - people doing or feeling something\\n"\n    "  4. editorial_text          - a claim or quote no footage can honestly show\\n"\n    "  5. stock_text              - a claim wanting atmospheric footage under it\\n"\n    "  6. stock_video / stock_photo - a real filmed moment: place, texture, mood\\n"\n    "  7. presenter               - the host addressing the viewer directly\\n"\n    "  8. t2v / broll             - last resort, when no stock clip could exist\\n"\n    "\\n"\n    "FIELD RULES\\n"\n    "- index: the zero-based position of the beat, copied from the cue list. "\n    "Every beat needs one, and they must be unique and in order.\\n"\n    "- stockRequirements: required for stock_video, stock_photo and stock_text. "\n    "Write queries the way a stock photographer searches — subject, action, "\n    "setting. \'expensive grocery shopping\' finds footage; \'inflation erodes "\n    "purchasing power\' finds nothing.\\n"\n    "- textOverlay: required for stock_text and editorial_text. Under 12 words. "\n    "It is a caption, not the narration repeated.\\n"\n    "- graphic.items: required for stickman, whiteboard, chart, map, timeline "\n    "and diagram, and it must carry real content pulled from the narration — "\n    "\'Label: Number\' for chart, \'Date: Event\' for timeline, place names for "\n    "map, steps for whiteboard, \'action:expression\' for stickman. A graphic "\n    "with no items renders as a blank card.\\n"\n    "- Only claim a chart, map or timeline when the narration actually contains "\n    "the numbers, places or dates to fill it. Otherwise choose footage.\\n"\n    "\\n"\n    "PACING\\n"\n    "- Never let more than four consecutive beats share a visualType.\\n"\n    "- Vary shot scale; back-to-back identical framing kills momentum.\\n"\n    "- Change the kind of visual every 30-40 seconds.\\n"\n    "- Use presenter sparingly: the hook, a section turn, the close.\\n"\n    "\\n"\n    "Put any contradiction in place, time of day or weather between "\n    "consecutive beats into `warnings`."\n)\n'
Path('director/prompts.py').write_text(SRC, encoding='utf-8')
print('  wrote director/prompts.py', len(SRC), 'bytes')

In [ ]:
# director/cache.py — generated from the package by build_notebook.py
SRC = '"""Disk cache for director results.\n\nA storyboard costs minutes of GPU on a T4, and re-running the same script\nduring development is the common case. Keyed on everything that changes the\nanswer, so a schema or model change misses rather than serving a stale shape.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nimport tempfile\nfrom typing import Any, Optional\n\nCACHE_DIR = os.environ.get("DIRECTOR_CACHE_DIR", "/tmp/director_cache")\n\n\ndef init_cache() -> None:\n    os.makedirs(CACHE_DIR, exist_ok=True)\n\n\ndef get_cache_key(script: str, style: str, model_id: str, schema_version: str, extra: str = "") -> str:\n    payload = "|".join([script, style, model_id, schema_version, extra])\n    return hashlib.sha256(payload.encode("utf-8")).hexdigest()\n\n\ndef _path(cache_key: str) -> str:\n    return os.path.join(CACHE_DIR, f"{cache_key}.json")\n\n\ndef get_cached_result(cache_key: str) -> Optional[Any]:\n    try:\n        with open(_path(cache_key), encoding="utf-8") as handle:\n            return json.load(handle)\n    except (FileNotFoundError, json.JSONDecodeError):\n        # A half-written file from an interrupted run should miss, not crash.\n        return None\n\n\ndef set_cached_result(cache_key: str, data: Any) -> None:\n    os.makedirs(CACHE_DIR, exist_ok=True)\n    # Write-then-rename so a reader never sees a partial file.\n    fd, tmp = tempfile.mkstemp(dir=CACHE_DIR, suffix=".tmp")\n    try:\n        with os.fdopen(fd, "w", encoding="utf-8") as handle:\n            json.dump(data, handle)\n        os.replace(tmp, _path(cache_key))\n    except Exception:\n        if os.path.exists(tmp):\n            os.unlink(tmp)\n        raise\n'
Path('director/cache.py').write_text(SRC, encoding='utf-8')
print('  wrote director/cache.py', len(SRC), 'bytes')

In [ ]:
# director/inference.py — generated from the package by build_notebook.py
SRC = '"""Thin client over the local vLLM OpenAI-compatible server."""\n\nfrom __future__ import annotations\n\nimport json\nimport re\nimport time\nfrom typing import Any, Dict, Iterable, List, Optional, Tuple\n\nfrom openai import OpenAI\n\nfrom .prompts import AETHER_SYSTEM_PROMPT, DIRECTOR_SYSTEM_PROMPT\nfrom .schema import get_json_schema\n\n_THINK_BLOCK = re.compile(r"<think>.*?</think>", re.DOTALL)\n\n\ndef strip_thinking(text: str) -> str:\n    """Drop a Qwen reasoning block. Also handles the unclosed, truncated case."""\n    text = _THINK_BLOCK.sub("", text or "")\n    # A response cut off by max_tokens mid-thought leaves a dangling <think>\n    # and no content at all. Returning the raw tail is more useful than "".\n    if "<think>" in text:\n        text = text.split("</think>")[-1] if "</think>" in text else text.split("<think>")[-1]\n    return text.strip()\n\n\nclass DirectorInference:\n    """Wraps the model with the three shapes AETHER asks for: chat, generate, plan."""\n\n    def __init__(self, model_id: str, port: int = 8000, timeout: float = 1800):\n        self.model_id = model_id\n        self.client = OpenAI(\n            base_url=f"http://localhost:{port}/v1",\n            api_key="sk-no-key-required",  # vLLM runs unauthenticated on localhost\n            timeout=timeout,\n            max_retries=0,\n        )\n\n    # -- helpers ----------------------------------------------------------\n\n    @staticmethod\n    def _with_system(messages: List[Dict[str, str]], system: str) -> List[Dict[str, str]]:\n        """Return a NEW list with a system turn in front.\n\n        Never mutate the caller\'s list: FastAPI hands us the parsed request\n        body, and inserting into it in place would corrupt the request object\n        on any retry.\n        """\n        if any(m.get("role") == "system" for m in messages):\n            return list(messages)\n        return [{"role": "system", "content": system}, *messages]\n\n    @staticmethod\n    def _thinking(enabled: bool) -> Dict[str, Any]:\n        """Qwen3 toggles reasoning through the chat template, not a sampling arg."""\n        return {"chat_template_kwargs": {"enable_thinking": bool(enabled)}}\n\n    # -- conversational ---------------------------------------------------\n\n    def chat(\n        self,\n        messages: List[Dict[str, str]],\n        temperature: float = 0.7,\n        max_tokens: int = 1024,\n        thinking: bool = False,\n        stream: bool = False,\n        **kwargs: Any,\n    ):\n        """A normal chat turn. Returns text, or a chunk iterator when streaming."""\n        response = self.client.chat.completions.create(\n            model=self.model_id,\n            messages=self._with_system(messages, AETHER_SYSTEM_PROMPT),\n            temperature=temperature,\n            max_tokens=max_tokens,\n            stream=stream,\n            extra_body=self._thinking(thinking),\n            **kwargs,\n        )\n        if stream:\n            return response\n        return strip_thinking(response.choices[0].message.content or "")\n\n    def generate(\n        self,\n        prompt: str,\n        temperature: float = 0.7,\n        max_tokens: int = 1024,\n        thinking: bool = False,\n        response_format: Optional[Dict[str, Any]] = None,\n        **kwargs: Any,\n    ) -> str:\n        """One-shot completion for the structured application tasks."""\n        if response_format is not None:\n            kwargs["response_format"] = response_format\n        return self.chat(\n            [{"role": "user", "content": prompt}],\n            temperature=temperature,\n            max_tokens=max_tokens,\n            thinking=thinking,\n            **kwargs,\n        )\n\n    # -- storyboard -------------------------------------------------------\n\n    def _plan_call(self, messages, max_tokens: int, mode: str):\n        """Ask for schema-constrained JSON.\n\n        vLLM has moved the structured-output knob twice, so rather than pin one\n        spelling and break on the next release, try them oldest-supported-last\n        and let the caller fall through. `mode` names which spelling to use.\n        """\n        schema = get_json_schema()\n        common = dict(\n            model=self.model_id,\n            messages=messages,\n            temperature=0.0,\n            max_tokens=max_tokens,\n        )\n\n        if mode == "response_format":\n            return self.client.chat.completions.create(\n                **common,\n                response_format={\n                    "type": "json_schema",\n                    "json_schema": {"name": "video_plan", "schema": schema},\n                },\n                extra_body=self._thinking(False),\n            )\n        if mode == "structured_outputs":\n            return self.client.chat.completions.create(\n                **common,\n                extra_body={\n                    "structured_outputs": {"json": schema},\n                    **self._thinking(False),\n                },\n            )\n        return self.client.chat.completions.create(\n            **common,\n            extra_body={"guided_json": schema, **self._thinking(False)},\n        )\n\n    def generate_plan(\n        self,\n        script: str,\n        style: str = "documentary",\n        title: str = "Untitled",\n        cues: Optional[List[Dict[str, Any]]] = None,\n        brief: str = "",\n        max_tokens: int = 6144,\n    ) -> Tuple[dict, float]:\n        """Produce a video plan in AETHER\'s `parseVideoPlan` shape.\n\n        Reasoning is forced off. The grammar makes the very first token an\n        opening brace, so a model that wants to think first has nowhere to put\n        the thought — enabling it makes the request fail, not deliberate.\n        """\n        beats = ""\n        if cues:\n            lines = []\n            for position, cue in enumerate(cues):\n                text = str(cue.get("text") or cue.get("subtitle") or "").strip()\n                # AETHER merges the answer back onto its own scenes by index, so\n                # its numbering wins whenever it sends one.\n                index = cue.get("index", position)\n                start = cue.get("start", cue.get("timestamp"))\n                stamp = f" (t={start}s)" if start is not None else ""\n                lines.append(f"  index {index}{stamp}: {text}")\n            beats = (\n                "\\n\\nBEATS — return exactly one scene for each line below, "\n                "reusing these index numbers exactly:\\n" + "\\n".join(lines)\n            )\n\n        messages = [\n            {"role": "system", "content": DIRECTOR_SYSTEM_PROMPT},\n            {\n                "role": "user",\n                "content": (\n                    f"Title: {title}\\n"\n                    f"Visual style: {style}\\n"\n                    + (f"\\n{brief}\\n" if brief else "")\n                    + f"\\nScript:\\n{script}"\n                    f"{beats}"\n                ),\n            },\n        ]\n\n        last_error: Optional[Exception] = None\n        started = time.time()\n        for mode in ("response_format", "guided_json", "structured_outputs"):\n            try:\n                response = self._plan_call(messages, max_tokens, mode)\n                break\n            except Exception as exc:  # noqa: BLE001 - reported if every mode fails\n                last_error = exc\n        else:\n            raise RuntimeError(\n                f"vLLM rejected every structured-output spelling. Last error: {last_error}"\n            )\n\n        latency = time.time() - started\n        raw = response.choices[0].message.content or ""\n        cleaned = strip_thinking(raw)\n        # The grammar should make this unreachable, but a truncated response\n        # (max_tokens hit mid-object) lands here, and the raw tail is the only\n        # useful thing to show.\n        try:\n            return json.loads(cleaned), latency\n        except json.JSONDecodeError as exc:\n            raise ValueError(\n                f"Model returned invalid JSON after {latency:.1f}s ({exc}). "\n                f"Finish reason: {response.choices[0].finish_reason}. "\n                f"Tail: ...{cleaned[-400:]!r}"\n            ) from exc\n\n    # Older notebooks call this name.\n    def generate_storyboard(self, script: str, style: str = "documentary", **kwargs):\n        kwargs.pop("reasoning_effort", None)  # retired; reasoning is always off here\n        return self.generate_plan(script=script, style=style, **kwargs)\n'
Path('director/inference.py').write_text(SRC, encoding='utf-8')
print('  wrote director/inference.py', len(SRC), 'bytes')

In [ ]:
# director/api.py — generated from the package by build_notebook.py
SRC = '"""FastAPI surface for AETHER.\n\nThree ways in, one model behind them: /chat for conversation, /generate for the\nstructured application tasks, /director for a storyboard against the grammar.\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport os\nfrom contextlib import asynccontextmanager\nfrom typing import Any, Dict, List, Optional\n\nfrom fastapi import Depends, FastAPI, HTTPException\nfrom fastapi.middleware.cors import CORSMiddleware\nfrom fastapi.responses import StreamingResponse\nfrom fastapi.security import HTTPAuthorizationCredentials, HTTPBearer\nfrom pydantic import BaseModel\n\nfrom .cache import get_cache_key, get_cached_result, init_cache, set_cached_result\nfrom .inference import DirectorInference\n\nAPI_KEY = os.environ.get("DIRECTOR_API_KEY", "test-key-change-me")\n# Only a fallback: the notebook exports DIRECTOR_MODEL before uvicorn starts,\n# so this is what you get running the API standalone. Kept in step with\n# MODEL_CHOICE in build_notebook.py, or /health reports a model that is not loaded.\nMODEL_ID = os.environ.get("DIRECTOR_MODEL", "Qwen/Qwen3-14B-AWQ")\nVLLM_PORT = int(os.environ.get("VLLM_PORT", "8000"))\nSCHEMA_VERSION = "2.0-aether"\n\n_engine: Optional[DirectorInference] = None\n\n\ndef get_engine() -> DirectorInference:\n    global _engine\n    if _engine is None:\n        _engine = DirectorInference(model_id=MODEL_ID, port=VLLM_PORT)\n    return _engine\n\n\n@asynccontextmanager\nasync def lifespan(_: FastAPI):\n    init_cache()\n    yield\n\n\napp = FastAPI(title="AETHER Qwen Brain", lifespan=lifespan)\n\n# The browser never reaches this directly — AETHER\'s node server proxies it —\n# but allowing the origin makes a direct curl or a tunnel probe behave.\napp.add_middleware(\n    CORSMiddleware,\n    allow_origins=["*"],\n    allow_methods=["*"],\n    allow_headers=["*"],\n)\n\nsecurity = HTTPBearer(auto_error=True)\n\n\ndef check_token(creds: HTTPAuthorizationCredentials = Depends(security)) -> str:\n    if creds.credentials != API_KEY:\n        raise HTTPException(401, "Invalid API key")\n    return creds.credentials\n\n\nclass ChatRequest(BaseModel):\n    messages: List[Dict[str, str]]\n    temperature: float = 0.7\n    max_tokens: int = 1024\n    thinking: bool = False\n    stream: bool = False\n\n\nclass GenerateRequest(BaseModel):\n    prompt: str\n    temperature: float = 0.7\n    max_tokens: int = 2048\n    thinking: bool = False\n    response_format: Optional[Dict[str, Any]] = None\n\n\nclass DirectorRequest(BaseModel):\n    script: str\n    title: str = "Untitled"\n    style: str = "documentary"\n    language: str = "en"\n    cues: Optional[List[Dict[str, Any]]] = None\n    brief: str = ""\n    max_tokens: int = 6144\n    no_cache: bool = False\n\n\n# /health is deliberately unauthenticated: AETHER polls it every few seconds to\n# decide whether Qwen is up, and a health probe that can fail on auth would\n# report the model down whenever the key is merely misconfigured.\n@app.get("/health")\ndef health() -> dict:\n    return {"status": "ok", "model": MODEL_ID, "schema_version": SCHEMA_VERSION}\n\n\n@app.get("/model")\ndef model_info() -> dict:\n    return {"model": MODEL_ID, "schema_version": SCHEMA_VERSION}\n\n\n@app.post("/chat")\ndef chat_endpoint(req: ChatRequest, _: str = Depends(check_token)):\n    engine = get_engine()\n\n    if not req.stream:\n        try:\n            text = engine.chat(\n                req.messages,\n                temperature=req.temperature,\n                max_tokens=req.max_tokens,\n                thinking=req.thinking,\n            )\n        except Exception as exc:  # noqa: BLE001\n            raise HTTPException(502, f"Model error: {exc}") from exc\n        return {"choices": [{"message": {"role": "assistant", "content": text}}]}\n\n    try:\n        chunks = engine.chat(\n            req.messages,\n            temperature=req.temperature,\n            max_tokens=req.max_tokens,\n            thinking=req.thinking,\n            stream=True,\n        )\n    except Exception as exc:  # noqa: BLE001\n        raise HTTPException(502, f"Model error: {exc}") from exc\n\n    def sse():\n        try:\n            for chunk in chunks:\n                if not chunk.choices:\n                    continue\n                delta = chunk.choices[0].delta.content or ""\n                if not delta:\n                    continue\n                payload = {"choices": [{"delta": {"content": delta}}]}\n                yield f"data: {json.dumps(payload)}\\n\\n"\n        except Exception as exc:  # noqa: BLE001\n            # The response has already begun, so the only way to report a\n            # mid-stream failure is inside the stream itself.\n            yield f"data: {json.dumps({\'error\': str(exc)})}\\n\\n"\n        yield "data: [DONE]\\n\\n"\n\n    return StreamingResponse(\n        sse(),\n        media_type="text/event-stream",\n        headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"},\n    )\n\n\n@app.post("/generate")\ndef generate_endpoint(req: GenerateRequest, _: str = Depends(check_token)) -> dict:\n    try:\n        text = get_engine().generate(\n            req.prompt,\n            temperature=req.temperature,\n            max_tokens=req.max_tokens,\n            thinking=req.thinking,\n            response_format=req.response_format,\n        )\n    except Exception as exc:  # noqa: BLE001\n        raise HTTPException(502, f"Model error: {exc}") from exc\n    return {"content": text}\n\n\n@app.post("/director")\ndef director_endpoint(req: DirectorRequest, _: str = Depends(check_token)) -> dict:\n    key = get_cache_key(req.script, req.style, MODEL_ID, SCHEMA_VERSION, f"{req.cues}|{req.brief}")\n    if not req.no_cache:\n        cached = get_cached_result(key)\n        if cached is not None:\n            return {"success": True, "cached": True, "plan": cached, **cached}\n\n    try:\n        plan, latency = get_engine().generate_plan(\n            script=req.script,\n            style=req.style,\n            title=req.title,\n            cues=req.cues,\n            brief=req.brief,\n            max_tokens=req.max_tokens,\n        )\n    except Exception as exc:  # noqa: BLE001\n        raise HTTPException(502, f"Director failed: {exc}") from exc\n\n    set_cached_result(key, plan)\n    # `plan` is also splatted at the top level so a caller can read `scenes`\n    # straight off the response without knowing about this envelope.\n    return {\n        "success": True,\n        "cached": False,\n        "latency_sec": round(latency, 2),\n        "plan": plan,\n        **plan,\n    }\n'
Path('director/api.py').write_text(SRC, encoding='utf-8')
print('  wrote director/api.py', len(SRC), 'bytes')

In [ ]:
# director/tests.py — generated from the package by build_notebook.py
SRC = '"""Contract tests. No GPU, no model — these run anywhere in about a second.\n\nThe thing worth testing here is not that pydantic works, it is that the schema\nstill lines up with what AETHER\'s parseVideoPlan() reads. That parser drops a\nscene with no numeric index and silently rewrites an unknown visualType, so a\ndrift between the two files produces an empty storyboard and no error.\n\n    python -m unittest director.tests -v\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport unittest\n\nfrom .cache import get_cache_key\nfrom .schema import VideoPlan, get_json_schema\n\n# Mirrors the vocabularies in public/prompts.js. If a test here fails after you\n# edit that file, the fix is to change both, not to loosen the test.\nAETHER_VISUAL_TYPES = {\n    "stock_video", "stock_photo", "stock_text", "editorial_text",\n    "t2v", "broll", "presenter",\n    "stickman", "whiteboard", "chart", "map", "timeline", "diagram",\n}\nAETHER_SCENE_FIELDS = {\n    "index", "visualType", "stockRequirements", "textOverlay", "graphic",\n    "hostOverlay", "shotType", "cameraMovement", "motion", "emotion",\n    "transition", "note",\n}\n\nMINIMAL_PLAN = {\n    "strategy": "Footage for the concrete beats, a chart for the one number.",\n    "warnings": [],\n    "scenes": [\n        {\n            "index": 0,\n            "visualType": "stock_video",\n            "stockRequirements": {\n                "concept": "A shopper faced with higher prices.",\n                "queries": ["woman shopping for groceries", "supermarket price label close up"],\n                "fallbackQueries": ["grocery store aisle"],\n                "subjectCategory": "HUMAN",\n                "minimumDuration": 3.0,\n            },\n            "shotType": "Medium",\n            "cameraMovement": "Slow Push In",\n            "motion": "She lifts an item and puts it back.",\n            "emotion": "resigned",\n            "transition": "cut",\n        },\n        {\n            "index": 1,\n            "visualType": "chart",\n            "graphic": {\n                "title": "Food prices",\n                "subtitle": "",\n                "items": ["2021: 61", "2024: 87"],\n            },\n            "transition": "dissolve",\n        },\n    ],\n}\n\n\nclass TestSchemaShape(unittest.TestCase):\n    def test_json_schema_is_object_with_scenes(self):\n        schema = get_json_schema()\n        self.assertEqual(schema["type"], "object")\n        self.assertIn("scenes", schema["properties"])\n\n    def test_objects_forbid_extra_keys(self):\n        """Guided decoding follows the grammar; an open object invites invention."""\n\n        def walk(node):\n            if isinstance(node, dict):\n                if node.get("type") == "object":\n                    self.assertIs(\n                        node.get("additionalProperties"), False,\n                        msg=f"object left open: {json.dumps(node)[:120]}",\n                    )\n                for value in node.values():\n                    walk(value)\n            elif isinstance(node, list):\n                for value in node:\n                    walk(value)\n\n        walk(get_json_schema())\n\n    def test_scene_fields_match_aether_parser(self):\n        fields = set(VideoPlan.model_json_schema()["$defs"]["Scene"]["properties"])\n        self.assertEqual(\n            fields, AETHER_SCENE_FIELDS,\n            "Scene fields drifted from parseVideoPlan() in public/prompts.js",\n        )\n\n    def test_visual_types_match_aether_vocabulary(self):\n        enum = set(VideoPlan.model_json_schema()["$defs"]["Scene"]["properties"]["visualType"]["enum"])\n        self.assertEqual(\n            enum, AETHER_VISUAL_TYPES,\n            "visualType drifted from VISUAL_TYPES in public/prompts.js",\n        )\n\n\nclass TestValidation(unittest.TestCase):\n    def test_minimal_plan_validates(self):\n        plan = VideoPlan(**MINIMAL_PLAN)\n        self.assertEqual(len(plan.scenes), 2)\n        self.assertEqual(plan.scenes[0].stockRequirements.queries[0], "woman shopping for groceries")\n\n    def test_defaults_fill_in(self):\n        """AETHER reads every scene field, so none of them may be absent."""\n        scene = VideoPlan(**MINIMAL_PLAN).scenes[1]\n        self.assertEqual(scene.hostOverlay, "none")\n        self.assertEqual(scene.shotType, "Medium")\n        self.assertEqual(scene.cameraMovement, "Static")\n\n    def test_index_is_required(self):\n        """A scene with no index is dropped by AETHER, so reject it here."""\n        broken = json.loads(json.dumps(MINIMAL_PLAN))\n        del broken["scenes"][0]["index"]\n        with self.assertRaises(Exception):\n            VideoPlan(**broken)\n\n    def test_unknown_visual_type_rejected(self):\n        broken = json.loads(json.dumps(MINIMAL_PLAN))\n        broken["scenes"][0]["visualType"] = "ai_generated_clip"\n        with self.assertRaises(Exception):\n            VideoPlan(**broken)\n\n    def test_empty_scenes_rejected(self):\n        with self.assertRaises(Exception):\n            VideoPlan(strategy="", warnings=[], scenes=[])\n\n\nclass TestCache(unittest.TestCase):\n    def test_key_is_stable(self):\n        args = ("script", "finance", "model-1", "2.0", "")\n        self.assertEqual(get_cache_key(*args), get_cache_key(*args))\n\n    def test_schema_version_changes_key(self):\n        a = get_cache_key("s", "finance", "m", "1.0", "")\n        b = get_cache_key("s", "finance", "m", "2.0", "")\n        self.assertNotEqual(a, b, "a schema change must miss the cache, not serve a stale shape")\n\n\nif __name__ == "__main__":\n    unittest.main()\n'
Path('director/tests.py').write_text(SRC, encoding='utf-8')
print('  wrote director/tests.py', len(SRC), 'bytes')

## Stage 5 — Compile and run the contract tests

No GPU needed. This is what catches the schema drifting away from AETHER's parser.

In [ ]:
import glob, py_compile, subprocess, sys

for path in sorted(glob.glob('director/*.py')):
    py_compile.compile(path, doraise=True)
    print('  compiles:', path)

result = subprocess.run([sys.executable, '-m', 'unittest', 'director.tests', '-v'],
                        capture_output=True, text=True)
print(result.stdout[-3000:])
print(result.stderr[-3000:])
if result.returncode != 0:
    raise RuntimeError('Contract tests failed — the schema no longer matches AETHER.')
print('Stage 5 PASSED')

## Stage 6 — Boot vLLM

First run downloads the weights, so allow 10-25 minutes. A heartbeat prints every 60s.

In [ ]:
import os, subprocess, sys, threading, time
import requests, torch

env = os.environ.copy()
# FlashInfer JIT-compiles kernels that Turing cannot use; turning it off avoids
# a long build that ends in an unsupported-arch error.
env['VLLM_USE_FLASHINFER_SAMPLER'] = '0'

cmd = [
    sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
    '--model',                  DIRECTOR_MODEL,
    '--tensor-parallel-size',   str(DIRECTOR_TP_SIZE),
    '--max-model-len',          str(DIRECTOR_MAX_LEN),
    '--dtype',                  DTYPE,
    '--gpu-memory-utilization', str(GPU_MEM_UTIL),
    '--max-num-seqs',           str(MAX_NUM_SEQS),
    '--enforce-eager',
    '--port',                   str(VLLM_PORT),
]
print(' '.join(cmd), '\n')

vllm_proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1, env=env)

def _drain():
    for line in iter(vllm_proc.stdout.readline, ''):
        print('  [vLLM]', line, end='', flush=True)

threading.Thread(target=_drain, daemon=True).start()

DEADLINE = time.time() + 1800   # 30 min: a cold 19 GB download is not quick
started, last_beat, booted = time.time(), time.time(), False

while time.time() < DEADLINE:
    time.sleep(5)

    if vllm_proc.poll() is not None:
        raise RuntimeError(
            f'vLLM exited with code {vllm_proc.returncode} before serving. '
            'Read the [vLLM] lines above — an unsupported-quantization or '
            'out-of-memory error will be near the end.'
        )

    try:
        if requests.get(f'http://localhost:{VLLM_PORT}/health', timeout=3).status_code == 200:
            booted = True
            break
    except Exception:
        pass

    if time.time() - last_beat >= 60:
        elapsed = int(time.time() - started)
        print(f'  [heartbeat] {elapsed//60}m{elapsed%60:02d}s — still loading...', flush=True)
        last_beat = time.time()

if not booted:
    vllm_proc.terminate()
    raise RuntimeError('vLLM did not become healthy within 30 minutes.')

print(f'\nvLLM healthy after {int(time.time()-started)}s')
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f'  GPU {i}: {(total-free)/1024**3:.1f}/{total/1024**3:.1f} GB used')
print('Stage 6 PASSED')

## Stage 7 — Smoke test

In [ ]:
import time
from openai import OpenAI

client = OpenAI(base_url=f'http://localhost:{VLLM_PORT}/v1', api_key='sk-no-key', timeout=600)

t0 = time.time()
resp = client.chat.completions.create(
    model=DIRECTOR_MODEL,
    messages=[{'role': 'user', 'content': 'Say exactly: INFERENCE OK'}],
    max_tokens=64,
    temperature=0.0,
    # Qwen3 opens with a <think> block unless told not to. With a small
    # max_tokens the reply would be all reasoning and no content, which reads
    # as an empty response rather than as the truncation it is.
    extra_body={'chat_template_kwargs': {'enable_thinking': False}},
)
text = (resp.choices[0].message.content or '').strip()
print(f'Response: {text!r}')
print(f'Latency : {time.time()-t0:.1f}s')
if not text:
    raise RuntimeError(f'Empty response (finish_reason={resp.choices[0].finish_reason}).')
print('Stage 7 PASSED')

## Stage 8 — Structured plan against the AETHER grammar

In [ ]:
import json
from director.inference import DirectorInference
from director.schema import VideoPlan

engine = DirectorInference(model_id=DIRECTOR_MODEL, port=VLLM_PORT)

TEST_SCRIPT = (
    'Inflation quietly reduces what your paycheck can buy over time. '
    'As prices rise, the same salary buys fewer groceries and less fuel. '
    'Food prices rose from an index of 61 in 2021 to 87 in 2024. '
    'Central banks respond by raising interest rates, which affects mortgages and spending.'
)

plan, latency = engine.generate_plan(script=TEST_SCRIPT, style='finance', title='Inflation explained')
validated = VideoPlan(**plan)

print(f'Latency : {latency:.1f}s')
print(f'Scenes  : {len(validated.scenes)}')
print(f'Strategy: {validated.strategy}\n')
for s in validated.scenes:
    detail = ''
    if s.stockRequirements:
        detail = ' | '.join(s.stockRequirements.queries[:2])
    elif s.graphic:
        detail = ', '.join(s.graphic.items[:3])
    elif s.textOverlay:
        detail = s.textOverlay.text
    print(f'  [{s.index}] {s.visualType:15} {detail}')

director_plan = plan
print('\nStage 8 PASSED')

## Stage 9 — AETHER compatibility

Re-implements what `parseVideoPlan()` does, so a plan that would arrive empty in the app fails here instead.

In [ ]:
VISUAL_TYPES = {
    'stock_video', 'stock_photo', 'stock_text', 'editorial_text',
    't2v', 'broll', 'presenter',
    'stickman', 'whiteboard', 'chart', 'map', 'timeline', 'diagram',
}
NEEDS_STOCK   = {'stock_video', 'stock_photo', 'stock_text'}
NEEDS_TEXT    = {'stock_text', 'editorial_text'}
NEEDS_GRAPHIC = {'stickman', 'whiteboard', 'chart', 'map', 'timeline', 'diagram'}

issues, warnings = [], []
scenes = director_plan.get('scenes', [])

if not scenes:
    issues.append('no scenes — AETHER would show an empty storyboard')

seen = set()
for s in scenes:
    idx = s.get('index')
    tag = f'scene[{idx}]'
    # AETHER drops any scene whose index is not a finite number.
    if not isinstance(idx, (int, float)) or isinstance(idx, bool):
        issues.append(f'{tag}: index missing or not numeric — this beat would be dropped')
    elif idx in seen:
        issues.append(f'{tag}: duplicate index')
    else:
        seen.add(idx)

    vt = s.get('visualType')
    if vt not in VISUAL_TYPES:
        issues.append(f'{tag}: unknown visualType {vt!r} — would silently become stock_video')

    queries = ((s.get('stockRequirements') or {}).get('queries')) or []
    if vt in NEEDS_STOCK and not queries:
        issues.append(f'{tag}: {vt} with no stock queries — nothing to search for')
    for q in queries:
        if len(str(q).split()) < 2:
            warnings.append(f'{tag}: one-word query {q!r} will match almost anything')

    if vt in NEEDS_TEXT and not ((s.get('textOverlay') or {}).get('text')):
        issues.append(f'{tag}: {vt} with no textOverlay.text — renders a blank card')

    if vt in NEEDS_GRAPHIC and not ((s.get('graphic') or {}).get('items')):
        issues.append(f'{tag}: {vt} with no graphic.items — renders a blank card')

print(f'{len(scenes)} scenes, {len(set(s.get("visualType") for s in scenes))} distinct visual types')
for w in warnings:
    print('  warn:', w)
if issues:
    print()
    for i in issues:
        print('  FAIL:', i)
    raise RuntimeError(f'{len(issues)} compatibility issue(s) — AETHER could not use this plan.')
print('\nStage 9 PASSED — plan is AETHER-compatible')

## Stage 10 — FastAPI + ngrok

In [ ]:
import subprocess, sys, time
import requests

api_log = open('/tmp/api.log', 'w')
api_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'director.api:app', '--host', '0.0.0.0', '--port', str(API_PORT)],
    stdout=api_log, stderr=subprocess.STDOUT,
)

health = None
for _ in range(30):
    time.sleep(2)
    try:
        health = requests.get(f'http://localhost:{API_PORT}/health', timeout=5).json()
        break
    except Exception:
        if api_proc.poll() is not None:
            print(open('/tmp/api.log').read()[-3000:])
            raise RuntimeError('uvicorn exited — see log above.')
if health is None:
    print(open('/tmp/api.log').read()[-3000:])
    raise RuntimeError('FastAPI never became healthy.')
print('health:', health)

AUTH = {'Authorization': f'Bearer {DIRECTOR_API_KEY}'}

# Auth must actually be enforced, and /health must actually be open.
assert requests.post(f'http://localhost:{API_PORT}/generate',
                     json={'prompt': 'hi'}, timeout=30).status_code in (401, 403), 'auth not enforced!'

chat = requests.post(f'http://localhost:{API_PORT}/chat', headers=AUTH,
                     json={'messages': [{'role': 'user', 'content': 'Reply with one word: ready'}],
                           'max_tokens': 32}, timeout=300)
print('/chat     ', chat.status_code, chat.json()['choices'][0]['message']['content'][:60] if chat.ok else chat.text[:200])

gen = requests.post(f'http://localhost:{API_PORT}/generate', headers=AUTH,
                    json={'prompt': 'Give one YouTube title about inflation.', 'max_tokens': 64}, timeout=300)
print('/generate ', gen.status_code, gen.json()['content'][:60] if gen.ok else gen.text[:200])

plan = requests.post(f'http://localhost:{API_PORT}/director', headers=AUTH,
                     json={'script': TEST_SCRIPT, 'style': 'finance'}, timeout=1800)
print('/director ', plan.status_code, f"{len(plan.json().get('scenes', []))} scenes" if plan.ok else plan.text[:200])
if not (chat.ok and gen.ok and plan.ok):
    raise RuntimeError('An endpoint failed — see the statuses above.')

PUBLIC_URL = None
if NGROK_AUTHTOKEN:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
    PUBLIC_URL = ngrok.connect(API_PORT).public_url
    print(f'\n  Public URL : {PUBLIC_URL}')
    print(f'  API key    : {DIRECTOR_API_KEY}')
    print('\n  Put these in AETHER\'s .env, then restart the node server:')
    print(f'    QWEN_API_URL={PUBLIC_URL}')
    print(f'    QWEN_API_KEY={DIRECTOR_API_KEY}')
else:
    print('\nNGROK_AUTHTOKEN not set — reachable on localhost only.')
print('\nStage 10 PASSED')

## Stage 11 — Multi-domain benchmark

Optional, and slow on a T4. Raise `N_DOMAINS` once you know the timings.

In [ ]:
import time

N_DOMAINS = 4   # of 12; each plan takes roughly 1-4 min on 2x T4

DOMAINS = [
    ('finance',      'Inflation quietly reduces what your paycheck can buy. Central banks raise rates to cool prices.'),
    ('history',      'In 1944, Allied forces launched the largest amphibious invasion in history at Normandy.'),
    ('science',      'DNA carries the instructions for life. Every cell holds the same three billion base pairs.'),
    ('technology',   'Language models are trained on billions of tokens and predict the next word from learned patterns.'),
    ('medicine',     'Sleeping fewer than six hours a night significantly raises the risk of heart disease.'),
    ('cooking',      'The secret to risotto is patience: add warm stock one ladle at a time, stirring constantly.'),
    ('education',    'The Socratic method asks students to question assumptions rather than absorb answers.'),
    ('fitness',      'Interval training burns more calories in twenty minutes than an hour of steady jogging.'),
    ('business',     'Startups focused on customer problems are far likelier to find product-market fit.'),
    ('psychology',   'Cognitive dissonance is the discomfort of holding two conflicting beliefs at once.'),
    ('geography',    'The Amazon produces a fifth of the world oxygen and hosts a tenth of all known species.'),
    ('storytelling', 'Every story is a character who wants something, obstacles, and what the struggle reveals.'),
][:N_DOMAINS]

print(f'{"domain":<14}{"status":<8}{"secs":>7}{"scenes":>8}{"types":>7}')
print('-' * 46)

rows = []
for domain, script in DOMAINS:
    try:
        plan, latency = engine.generate_plan(script=script, style=domain)
        VideoPlan(**plan)
        scenes = plan['scenes']
        kinds = len({s['visualType'] for s in scenes})
        print(f'{domain:<14}{"ok":<8}{latency:>7.1f}{len(scenes):>8}{kinds:>7}')
        rows.append(True)
    except Exception as exc:
        print(f'{domain:<14}{"FAIL":<8}  {str(exc)[:40]}')
        rows.append(False)

print(f'\n{sum(rows)}/{len(rows)} passed')